# AI-Chatbot-LineOA — ภาพรวมโปรเจค

Tutorial นี้จะอธิบายการทำงานของโปรเจค AI Chatbot บน LINE Official Account
โดยใช้ **FastAPI + LangGraph + LINE Messaging API**

---

## Tech Stack

| Component | Technology | หน้าที่ |
|-----------|-----------|--------|
| Web Framework | **FastAPI** | รับ webhook จาก LINE |
| AI Orchestration | **LangGraph** | จัดการ flow การตอบคำถาม |
| LLM | **OpenAI GPT-4o-mini** | ตีความ intent และสร้างคำตอบ |
| LINE Integration | **line-bot-sdk** | ส่ง/รับข้อความ LINE |
| Session Store | **Memory / Redis** | เก็บ state ระหว่าง conversation |

---

## สถาปัตยกรรมโดยรวม (High-Level Architecture)

```
┌─────────────────────────────────────────────────────────────┐
│                      LINE Platform                          │
│   User sends message ──► LINE OA ──► Webhook (HTTPS POST)  │
└──────────────────────────────┬──────────────────────────────┘
                               │ POST /webhook-dev
                               │ POST /webhook-official
                               ▼
┌─────────────────────────────────────────────────────────────┐
│                     FastAPI Application                     │
│  ┌──────────────┐  ┌──────────────┐  ┌──────────────────┐  │
│  │ webhook_     │  │  LINE Event  │  │  Loading         │  │
│  │ router.py    │─►│  Extractor   │  │  Animation       │  │
│  └──────────────┘  └──────┬───────┘  └──────────────────┘  │
└─────────────────────────── │ ────────────────────────────────┘
                             │
                             ▼
┌─────────────────────────────────────────────────────────────┐
│                    LINE Handler Layer                       │
│  load_session() ──► run_ai_runtime() ──► save_session()    │
└──────────────────────────────┬──────────────────────────────┘
                               │
                               ▼
┌─────────────────────────────────────────────────────────────┐
│                   LangGraph Agent Graph                     │
│                                                             │
│  session_gate ──► intent_classify ──► [Module Node]        │
│                ├──► resume_router  ──► [Module Node]        │
│                └──► handoff_mode                            │
│                                                             │
│  Modules: hospital_info │ nurse │ package │ concierge       │
└──────────────────────────────┬──────────────────────────────┘
                               │
                               ▼
┌─────────────────────────────────────────────────────────────┐
│                    Session Store                            │
│         Memory (dev) ──── Redis (production)               │
└─────────────────────────────────────────────────────────────┘
```

---

## โครงสร้างโฟลเดอร์

```
AI-Chatbot-LineOA/
├── src/
│   ├── app/                     # FastAPI layer
│   │   ├── webhook_router.py    # Endpoints & request handling
│   │   ├── ai_runtime.py        # Bridge FastAPI ↔ LangGraph
│   │   ├── app_config.py        # Constants & defaults
│   │   ├── helpers/app_state.py # Semaphore (concurrency)
│   │   └── startup/             # Startup initialization
│   │
│   ├── agent/                   # LangGraph layer
│   │   ├── graph_stub.py        # Main graph definition
│   │   ├── shared/              # Shared utilities
│   │   │   ├── state_stub.py    # AgentState TypedDict
│   │   │   └── llm_factory.py   # LLM provider factory
│   │   └── stub_nodes/          # Graph nodes
│   │       ├── session_gate_stub.py
│   │       ├── intent_router/
│   │       ├── hospital_info_agent_module/
│   │       ├── nurse_agent_module/
│   │       ├── package_agent_module/
│   │       └── concierge_agent_module/
│   │
│   ├── integrations/line/       # LINE SDK wrappers
│   │   ├── handler_stub.py      # Main event handler
│   │   ├── messaging.py         # Send messages
│   │   ├── profile.py           # Get user profile
│   │   └── utils.py             # Event extraction
│   │
│   └── infra/                   # Infrastructure
│       ├── session_model.py     # Session dataclass
│       └── session_store/       # Storage backends
│
├── data/
│   └── FAQ_hospital_info.csv    # Hospital knowledge base
└── pyproject.toml
```

---

## Notebooks ใน Tutorial นี้

| Notebook | หัวข้อ | ความสำคัญ |
|----------|--------|----------|
| `01_lineoa_webhook.ipynb` | LINE OA + Webhook basics | ⭐⭐⭐ |
| `02_fastapi_setup.ipynb` | FastAPI app, endpoints, lifespan | ⭐⭐⭐ |
| `03_langgraph_basics.ipynb` | LangGraph concepts (State, Graph, Nodes) | ⭐⭐⭐ |
| `04_agent_graph.ipynb` | Graph routing & module nodes | ⭐⭐⭐ |
| `05_session_memory.ipynb` | Session management & persistence | ⭐⭐ |
| `06_end_to_end_flow.ipynb` | Full request trace ตั้งแต่ต้นจนจบ | ⭐⭐⭐ |


## ติดตั้ง Dependencies

```bash
# สร้าง virtual environment
python -m venv .venv
source .venv/bin/activate  # Windows: .venv\Scripts\activate

# ติดตั้ง dependencies จาก pyproject.toml
pip install -e .

# หรือติดตั้งแยก
pip install fastapi uvicorn langgraph langchain langchain-openai line-bot-sdk redis python-dotenv
```

## Environment Variables ที่จำเป็น

```bash
# .env file
LINE_CHANNEL_ACCESS_TOKEN_DEV=your_dev_token
LINE_CHANNEL_ACCESS_TOKEN_OFFICIAL=your_official_token
OPENAI_API_KEY=your_openai_key

# Optional
SESSION_STORE_BACKEND=memory   # หรือ redis
REDIS_URL=redis://localhost:6379/0
MAX_CONCURRENT_EVENTS=1
LOG_LEVEL=INFO
```